In [2]:
from huggingface_hub import login
# Paste your Hugging Face token (with `Read` access)
login("hf token")

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json

In [ ]:
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"  # adjust if needed

# MODEL_ID = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    # dtype=torch.float16,
    device_map="auto"
)

In [5]:
def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

In [42]:
### final prompt

SYSTEM_PROMPT = """ Your are a researcher. In this study, you are tasked with identifying similar stories.
You will be presented with three stories, a base, and two choices,a and b. 
You are to determine which of the candidate stories, a and b, is the most similar to the base.
Specifically, you will consider the stories’ narrative similarity.
It is an important task so read each story twice.
"""

In [ ]:
def compare_stories(base_story, story_a, story_b, max_new_tokens=350):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"""
        Base Story:
        \"\"\"{base_story}\"\"\"
        
        Story A:
        \"\"\"{story_a}\"\"\"
        
        Story B:
        \"\"\"{story_b}\"\"\"
        """
        }
    ]

    prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0,
            top_p=1,
            do_sample=False,
        )
    
    prompt_len = inputs["input_ids"].shape[1]
    
    response = tokenizer.decode(
        outputs[0][prompt_len:],
        skip_special_tokens=True
    ).strip()
    
    return response


# Example usage
if __name__ == "__main__":
    results = []
    for row in load_jsonl("test_track_a.jsonl"):
        base_story = row["anchor_text"]
        story_a = row["text_a"]
        story_b = row["text_b"]
    
        output = compare_stories(
            base_story=base_story,
            story_a=story_a,
            story_b=story_b
        )
        # print(output)
        results.append({
            "anchor_text": row.get("anchor_text"),
            "text_a": row.get("text_a"),
            "text_b": row.get("text_b"),
            "gold_a_is_closer": row.get("text_a_is_closer"),
            "model_output": output
        })
    # print(results)


In [44]:
import pandas as pd
pd.DataFrame(results).to_csv("predictionsFinal.csv", index=False)
# pd.DataFrame(results).to_json("TESTresults_llama_fewshot3_task4_v1.json", orient="records", indent=2)